In [ ]:
#cell 1: imports and setup               
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

plt.style.use('dark_background')
sns.set_palette('husl')

print("libraries loaded")


In [ ]:
#cell 2: load SST-2 from huggingFace

print("Dataset splits:",dataset)
print(f"\nTrain size:  {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")
print(f"\nColumns: {dataset['train'].column_names}")
print(f"\nLabel mapping: 0=negative, 1=positive")


In [ ]:
#cell 3: convert to pandas dataframe for easier exploration 
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()

#rename label column for clarity
train_df['sentiment']= train_df['label'].map({0:'negative',1:'positive'})
val_df['sentiment']= val_df['label'].map({0:'negative',1:'positive'})

#add text length column for analysis
train_df['text_length'] = train_df['sentence'].str.len()
train_df['word_count'] = train_df['sentence'].str.split().str.len()

print("train sample;")
train_df.head(5)




In [ ]:
# Cell 4: Class distribution — check for imbalance
counts = train_df['sentiment'].value_counts()
pct    = train_df['sentiment'].value_counts(normalize=True) * 100

print("=== CLASS DISTRIBUTION ===")
for label in counts.index:
    bar = "█" * int(pct[label] / 2)
    print(f"{label:10s}: {counts[label]:6,} ({pct[label]:.1f}%) {bar}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#3fb950', '#f85149']
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Class Distribution (Train)', fontsize=13)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Class Split', fontsize=13)

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to results/class_distribution.png")

In [ ]:
# Cell 5: Text length distribution by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color in [('positive', '#3fb950'), ('negative', '#f85149')]:
    subset = train_df[train_df['sentiment'] == label]
    axes[0].hist(subset['word_count'], bins=30, alpha=0.6,
                label=label, color=color)
    axes[1].hist(subset['text_length'], bins=30, alpha=0.6,
                label=label, color=color)

axes[0].set_title('Word Count by Sentiment')
axes[0].set_xlabel('Words'); axes[0].legend()
axes[1].set_title('Char Length by Sentiment')
axes[1].set_xlabel('Characters'); axes[1].legend()

plt.tight_layout()
plt.savefig('../results/length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== TEXT LENGTH STATS ===")
print(train_df.groupby('sentiment')[['word_count', 'text_length']].describe().round(1))

In [ ]:
# Cell 6: EDA summary — write your findings here
pos_count = counts['positive']
neg_count = counts['negative']
imbalance_ratio = pos_count / neg_count

summary = f"""
=== EDA FINDINGS — SST-2 ===

Dataset size:
  Train:      {len(train_df):,} samples
  Validation: {len(val_df):,} samples

Class balance:
  Positive: {pos_count:,} ({pct['positive']:.1f}%)
  Negative: {neg_count:,} ({pct['negative']:.1f}%)
  Ratio:    {imbalance_ratio:.2f}:1

Text characteristics (train):
  Avg word count:  {train_df['word_count'].mean():.1f}
  Median words:    {train_df['word_count'].median():.0f}
  Max words:       {train_df['word_count'].max()}
  Min words:       {train_df['word_count'].min()}

Key observations:
  - SST-2 is {'near-balanced' if 0.9 < imbalance_ratio < 1.1 else 'slightly imbalanced'}
  - Short texts (movie review snippets, not full reviews)
  - LLM prompts should handle short fragments well
  - Saved charts to results/

Next steps:
  - Open GitHub issue if notable imbalance found
  - Build preprocessing pipeline (Day 3)
  - Baseline model on these splits (Day 4)
"""
print(summary)

# Save findings to file
with open('../results/eda_findings.txt', 'w') as f:
    f.write(summary)
print("Findings saved to results/eda_findings.txt")